In [1]:
import os
import torch
import time
import re


# 0. OTIMIZAÇÃO PARA CPU
numero_de_nucleos = "8" 
os.environ["OMP_NUM_THREADS"] = numero_de_nucleos
os.environ["MKL_NUM_THREADS"] = numero_de_nucleos
torch.set_num_threads(int(numero_de_nucleos))

import langchain
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.output_parsers import StrOutputParser
from sentence_transformers import CrossEncoder

# NOVO: Imports para a Busca Híbrida
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document


# 1. CARREGAMENTO DOS MODELOS E BASES
load_dotenv()
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API")

print("\n# Processo iniciado em:", time.strftime("%d/%m/%Y - %H:%M:%S"))


# Processo iniciado em: 01/06/2026 - 16:15:31


In [21]:
# 1.1 Modelo Bi-Encoder
embedding = HuggingFaceEmbeddings(
    model_name='intfloat/multilingual-e5-large-instruct',
    model_kwargs={"device": "cpu", "trust_remote_code": True},
    encode_kwargs={"normalize_embeddings": True, "prompt": "query: "} 
)

# 1.2 Modelo Cross-Encoder (Reranker)
rerank_model = ['cross-encoder/mmarco-mMiniLMv2-L12-H384-v1','BAAI/bge-reranker-v2-m3']

print("Carregando Reranker:", rerank_model[0])
# Vamos carregar apenas o bge-m3 para este exemplo
reranker = CrossEncoder(rerank_model[0], device='cpu') 

def carregar_bases():
    bases = {
        "lotr"        : Chroma(persist_directory='db_e5_multilingual_li_Lor',              embedding_function=embedding),
        "contos"      : Chroma(persist_directory='db_e5_multilingual_li_Unfinished Tales', embedding_function=embedding),
        "silmarillion": Chroma(persist_directory='db_e5_multilingual_li_Silmarillion',     embedding_function=embedding),
    }
    total = sum(b._collection.count() for b in bases.values())
    print(f"✅ {len(bases)} bases carregadas | {total} chunks no total")
    return bases

bases = carregar_bases()

# NOVO: Função de tratamento de texto para remover pontuação
def preprocess_portugues(texto: str) -> list[str]:
    # Remove todos os caracteres especiais/pontuação e passa para minúsculo
    texto_limpo = re.sub(r'[^\w\s]', '', texto.lower())
    return texto_limpo.split()


# NOVO: Função para extrair dados do Chroma e montar o BM25
def construir_indice_bm25(bases):
    print("⏳ Construindo índice lexical (BM25)...")
    todos_documentos = []
    
    for nome, db in bases.items():
        # .get() pega todos os dados do banco
        dados = db.get()
        textos = dados['documents']
        metadatas = dados['metadatas']
        
        # Recria os objetos Document para o BM25
        for texto, meta in zip(textos, metadatas):
            todos_documentos.append(Document(page_content=texto, metadata=meta))
            
   # Passamos a nossa função customizada no parâmetro preprocess_func
    bm25_retriever = BM25Retriever.from_documents(
        todos_documentos, 
        preprocess_func=preprocess_portugues
    )
    bm25_retriever.k = 10  
    print("✅ Índice BM25 pronto e protegido contra pontuação!") 
    print("✅ Índice BM25 pronto!")
    return bm25_retriever

# Instancia o BM25 na inicialização do sistema
bm25_retriever = construir_indice_bm25(bases)


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Carregando Reranker: cross-encoder/mmarco-mMiniLMv2-L12-H384-v1


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ 3 bases carregadas | 8014 chunks no total
⏳ Construindo índice lexical (BM25)...
✅ Índice BM25 pronto e protegido contra pontuação!
✅ Índice BM25 pronto!


In [22]:
# 2. Função RAG Otimizada com Busca Híbrida
def rag_tolkien(pergunta: str, reranker_model) -> str:
    
    # 2.1 ESTÁGIO 1: RETRIEVAL HÍBRIDO (Semântica + Lexical)
    resultados_brutos = []
    
    # A) Busca Lexical (Palavras exatas via BM25)
    resultados_lexicais = bm25_retriever.invoke(pergunta)
    resultados_brutos.extend(resultados_lexicais)
    
    # B) Busca Semântica (Vetorial via Chroma)
    for nome, db in bases.items():
        resultados_brutos.extend(db.similarity_search(pergunta, k=5))
    
    # C) Remover Duplicatas (O BM25 e o Chroma podem trazer os mesmos chunks)
    docs_unicos = {}
    for doc in resultados_brutos:
        # Usa o conteúdo do texto como chave para garantir que não haja repetição
        if doc.page_content not in docs_unicos:
            docs_unicos[doc.page_content] = doc
            
    todos_resultados = list(docs_unicos.values())
    
    if not todos_resultados:
         return "Não encontrei informações preliminares sobre isso."

    # 2.2 ESTÁGIO 2: RERANKING (Mantido igual)
    pares_para_rerank = [[pergunta, doc.page_content] for doc in todos_resultados]
    notas_reranker = reranker_model.predict(pares_para_rerank)
    
    resultados_com_nota = list(zip(todos_resultados, notas_reranker))
    resultados_com_nota.sort(key=lambda x: x[1], reverse=True)
    
    top_5_finais = resultados_com_nota[:5]

    # 2.3 Monta contexto com referências (Mantido igual)
    context_parts = []
    for doc, score in top_5_finais:
        meta     = doc.metadata
        source   = meta.get('source',   '')
        volume   = meta.get('volume',   '')
        livro    = meta.get('livro',    '')
        parte    = meta.get('parte',    '')
        capitulo = meta.get('capitulo', '')
        secao    = meta.get('secao',    '')

        partes_ref = [p for p in [source, volume, livro, parte, capitulo, secao] if p]
        referencia = " | ".join(partes_ref)
        
        context_parts.append(f"[{referencia}]\n{doc.page_content}")
    
    context_text = "\n\n".join(context_parts)

    # 2.4 Template (Mantido igual)
    template = PromptTemplate.from_template(
        """
    Você é um assistente especialista na literatura de J.R.R. Tolkien.
    Responda à pergunta do usuário utilizando RESTRITAMENTE o contexto fornecido abaixo. 
    Se a resposta não puder ser encontrada no contexto, diga honestamente: "Não encontrei essa informação nos documentos fornecidos." - Não invente fatos.

    Diretrizes de Comportamento:
    1. Seja educado, objetivo e direto nas respostas gerais.
    2. Para perguntas sobre PERSONAGENS: Elabore uma resposta rica e detalhada, estruturando-a com base na raça do personagem, eventos históricos, fatos marcantes e suas consequências na história.
    3. Para perguntas sobre a BASE DE DADOS OU FONTES: Seja estritamente objetivo. Liste apenas o nome dos livros encontrados no contexto, sem detalhar ou resumir o conteúdo deles.(use os metadatas)

    Formatação Obrigatória:
    Ao final de QUALQUER resposta, cite as fontes utilizadas exatamente neste formato:
    📖 Referências: Volume | Livro | Capítulo (Ordene da mais relevante para a menos relevante, uma por linha)

    ## CONTEXTO
    {contexto}

    ## PERGUNTA DO USUÁRIO
    {pergunta}
        """
    )

    # 2.5 LLM Call (Mantido igual)
    llm_models = ['models/gemini-3.1-flash-lite', 'models/gemini-2.5-flash-lite', 'models/gemini-2.0-flash']
    for model_name in llm_models:
        try:
            llm   = ChatGoogleGenerativeAI(model=model_name, temperature=0.4, max_retries=2)
            chain = template | llm | StrOutputParser()
            return chain.invoke({"contexto": context_text, "pergunta": pergunta})
        except Exception as e:
            print(f"Modelo {model_name} falhou. Tentando próximo... Erro: {e}")
            continue

    return "Desculpe, todos os serviços de IA estão instáveis no momento. Tente novamente em instantes."

print("# Função RAG carregada:", time.strftime("%H:%M:%S"))


# Função RAG carregada: 16:51:55


In [23]:
# Testando agora com "Annatar", que falhou anteriormente
print("# inicio:", time.strftime("%H:%M:%S"))

resposta = rag_tolkien("Quem foi Annatar?", reranker)
print(resposta)
print("# fim:", time.strftime("%H:%M:%S"))


# inicio: 16:52:05
Annatar, cujo nome significa "Senhor dos Presentes", foi a identidade assumida por Sauron durante a Segunda Era.

**Detalhes sobre o personagem:**

*   **Raça:** Maia (embora tenha se disfarçado sob uma aparência bela e sábia para enganar os povos da Terra-média).
*   **Eventos Históricos e Atuação:** Sauron, sob o disfarce de Annatar, buscou influenciar os povos da Terra-média, sabendo que os elfos possuíam maior poder. Ele foi recebido com prazer em diversas regiões, onde os elfos tiraram proveito de sua amizade e conhecimentos. Contudo, sua presença foi rejeitada em Lindon, onde Gil-galad e Elrond duvidavam de sua natureza e impediam sua entrada.
*   **Fatos Marcantes:** Sua maior influência ocorreu em Eregion, onde logrou os artífices noldor, especialmente Celebrimbor. O desejo de Celebrimbor de igualar a habilidade e a fama de Fêanor facilitou as maquinações de Sauron. Enquanto isso, Galadriel trabalhava ativamente para neutralizar as ações de Annatar em Lórinan

In [24]:
# 2. Função RAG Otimizada com Busca Híbrida e Otimização de CPU
def rag_tolkien2(pergunta: str, reranker_model) -> str:
    
    # 2.1 ESTÁGIO 1: RETRIEVAL HÍBRIDO (Semântica + Lexical)
    resultados_brutos = []
    
    # A) Busca Lexical (Palavras exatas via BM25)
    resultados_lexicais = bm25_retriever.invoke(pergunta)
    resultados_brutos.extend(resultados_lexicais)
    
    # 🔥 SOLUÇÃO GARGALO 1: Calcula o embedding na CPU APENAS UMA VEZ
    vetor_pergunta = embedding.embed_query(pergunta)
    
    # B) Busca Semântica (Vetorial via Chroma - Usando o vetor pronto)
    for nome, db in bases.items():
        # Alterado para 'similarity_search_by_vector' (evita recalcular o embedding)
        # Reduzido k de 5 para 4 para enviar menos volume ao Reranker
        resultados_brutos.extend(db.similarity_search_by_vector(vetor_pergunta, k=4))
    
    # C) Remover Duplicatas
    docs_unicos = {}
    for doc in resultados_brutos:
        if doc.page_content not in docs_unicos:
            docs_unicos[doc.page_content] = doc
            
    todos_resultados = list(docs_unicos.values())
    
    if not todos_resultados:
         return "Não encontrei informações preliminares sobre isso."

    # 2.2 ESTÁGIO 2: RERANKING
    pares_para_rerank = [[pergunta, doc.page_content] for doc in todos_resultados]
    
    # O modelo gera uma lista de notas (logits) correspondente a cada par
    notas_reranker = reranker_model.predict(pares_para_rerank)
    
    resultados_com_nota = list(zip(todos_resultados, notas_reranker))
    resultados_com_nota.sort(key=lambda x: x[1], reverse=True)
    
    # Selecionamos apenas os 5 melhores absolutos após a reordenação
    top_5_finais = resultados_com_nota[:5]

    # 2.3 Monta contexto com referências (Mantido igual)
    context_parts = []
    for doc, score in top_5_finais:
        meta     = doc.metadata
        source   = meta.get('source',   '')
        volume   = meta.get('volume',   '')
        livro    = meta.get('livro',    '')
        parte    = meta.get('parte',    '')
        capitulo = meta.get('capitulo', '')
        secao    = meta.get('secao',    '')

        partes_ref = [p for p in [source, volume, livro, parte, capitulo, secao] if p]
        referencia = " | ".join(partes_ref)
        
        context_parts.append(f"[{referencia}]\n{doc.page_content}")
    
    context_text = "\n\n".join(context_parts)

    # 2.4 Template (Mantido igual)
    template = PromptTemplate.from_template(
        """
    Você é um assistente especialista na literatura de J.R.R. Tolkien.
    Responda à pergunta do usuário utilizando RESTRITAMENTE o contexto fornecido abaixo. 
    Se a resposta não puder ser encontrada no contexto, diga honestamente: "Não encontrei essa informação nos documentos fornecidos." - Não invente fatos.

    Diretrizes de Comportamento:
    1. Seja educado, objetivo e direto nas respostas gerais.
    2. Para perguntas sobre PERSONAGENS: Elabore uma resposta rica e detalhada, estruturando-a com base na raça do personagem, eventos históricos, fatos marcantes e suas consequências na história.
    3. Para perguntas sobre a BASE DE DADOS OU FONTES: Seja estritamente objetivo. Liste apenas o nome dos livros encontrados no contexto, sem detalhar ou resumir o conteúdo deles.(use os metadatas)

    Formatação Obrigatória:
    Ao final de QUALQUER resposta, cite as fontes utilizadas exatamente neste formato:
    📖 Referências: Volume | Livro | Capítulo (Ordene da mais relevante para a menos relevante, uma por linha)

    ## CONTEXTO
    {contexto}

    ## PERGUNTA DO USUÁRIO
    {pergunta}
        """
    )

    # 2.5 LLM Call (Mantido igual)
    llm_models = ['models/gemini-3.1-flash-lite', 'models/gemini-2.5-flash-lite', 'models/gemini-2.0-flash']
    for model_name in llm_models:
        try:
            llm   = ChatGoogleGenerativeAI(model=model_name, temperature=0.4, max_retries=2)
            chain = template | llm | StrOutputParser()
            return chain.invoke({"contexto": context_text, "pergunta": pergunta})
        except Exception as e:
            print(f"Modelo {model_name} falhou. Tentando próximo... Erro: {e}")
            continue

    return "Desculpe, todos os serviços de IA estão instáveis no momento. Tente novamente em instantes."

print("# Função RAG2 Otimizada carregada:", time.strftime("%H:%M:%S"))

# Função RAG2 Otimizada carregada: 16:52:22


In [25]:
# Testando agora com "Annatar", que falhou anteriormente
print("# inicio:", time.strftime("%H:%M:%S"))

resposta = rag_tolkien2("Quem foi Annatar?", reranker)
print(resposta)
print("# fim:", time.strftime("%H:%M:%S"))

# inicio: 16:52:28
Annatar, cujo nome significa "Senhor dos Presentes", foi a identidade assumida por Sauron durante a Segunda Era.

**Detalhes sobre o personagem:**

*   **Raça:** Maia (embora tenha se apresentado aos elfos sob uma forma física bela e sábia).
*   **Eventos Históricos e Fatos Marcantes:**
    *   Sauron buscou convencer os elfos a lhe prestarem serviço, ciente de seu grande poder. Para isso, adotou a aparência de Annatar e viajou pela Terra-média.
    *   Ele foi recebido com prazer por diversos povos élficos, que tiraram proveito de sua amizade e conhecimentos iniciais.
    *   Sua entrada foi proibida em Lindon por Gil-galad e Elrond, que desconfiavam de sua verdadeira natureza e de sua aparência.
    *   Obteve sucesso em manipular os noldor de Eregion, especialmente Celebrimbor, que desejava igualar a fama e habilidade de Fëanor.
*   **Consequências na História:** A influência de Annatar sobre os artífices de Eregion foi fundamental para as maquinações de Sauron, l